In [1]:
import os
import time
import requests
import pandas as pd
from IPython.display import display

TRINO_URL = os.getenv("TRINO_URL", "http://trino:8080")
TRINO_USER = os.getenv("TRINO_USER", "jupyter")

print("TRINO_URL =", TRINO_URL)
print("TRINO_USER =", TRINO_USER)

TRINO_URL = http://trino:8080
TRINO_USER = jupyter


In [2]:
def trino_query(sql_text: str, catalog: str | None = None, schema: str | None = None) -> pd.DataFrame:
    headers = {"X-Trino-User": TRINO_USER}
    if catalog:
        headers["X-Trino-Catalog"] = catalog
    if schema:
        headers["X-Trino-Schema"] = schema

    response = requests.post(
        f"{TRINO_URL}/v1/statement",
        data=sql_text.encode("utf-8"),
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    columns = payload.get("columns")
    rows = payload.get("data", [])

    while payload.get("nextUri"):
        response = requests.get(payload["nextUri"], timeout=30)
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            error = payload["error"]
            message = error.get("message", "Unknown Trino error")
            error_name = error.get("errorName", "TRINO_ERROR")
            raise RuntimeError(f"{error_name}: {message}")

        columns = columns or payload.get("columns")
        rows.extend(payload.get("data", []))

    if "error" in payload:
        error = payload["error"]
        message = error.get("message", "Unknown Trino error")
        error_name = error.get("errorName", "TRINO_ERROR")
        raise RuntimeError(f"{error_name}: {message}")

    if not columns:
        return pd.DataFrame(rows)

    column_names = [column["name"] for column in columns]
    return pd.DataFrame(rows, columns=column_names)


def show(sql_text: str, catalog: str | None = None, schema: str | None = None) -> pd.DataFrame:
    df = trino_query(sql_text, catalog=catalog, schema=schema)
    display(df)
    return df


In [10]:
query = """
WITH dedup_clickhouse AS (
    SELECT 
        from_utf8(rocket_name) AS rocket_name,
        launch_date,
        total_launches,
        success_rate,
        ROW_NUMBER() OVER(PARTITION BY launch_date, rocket_name ORDER BY processed_at DESC) AS rn
    FROM clickhouse.gold.rocket_success_rate
),
dedup_silver AS (
    SELECT 
        name AS mission_name,
        net AS launch_time,
        rocket_name,
        load_date,
        -- Оставляем только одну (самую свежую) запись для каждой миссии
        ROW_NUMBER() OVER(PARTITION BY name, net ORDER BY ingested_at DESC) AS rn_s
    FROM iceberg.silver.launches
)
SELECT 
    s.mission_name,
    s.launch_time,
    c.total_launches,
    c.success_rate
FROM dedup_silver s
JOIN dedup_clickhouse c
  ON s.rocket_name = c.rocket_name
 AND s.load_date = c.launch_date
WHERE s.rn_s = 1
  AND c.rn = 1
ORDER BY s.launch_time DESC
LIMIT 10
"""

show(query)

,mission_name,launch_time,total_launches,success_rate
0,Long March 8A | SatNet LEO Group 28,2026-10-25 02:03:00.000 UTC,1,100.0
1,Falcon 9 Block 5 | Starlink Group 15-15,2026-10-23 22:48:44.000 UTC,2,100.0
2,Long March 4B | Yaogan 53-07 to 09/56-19 to 21,2026-10-23 19:25:00.000 UTC,2,100.0
3,Long March 6A | Yaogan 50-03,2026-10-23 10:15:00.000 UTC,2,100.0
4,Electron | We're Not Hawking Around (3x HawkEy...,2026-10-23 07:11:00.000 UTC,2,100.0
5,Ceres-1S | Xiguang-4 01,2026-10-21 03:39:00.000 UTC,2,100.0
6,Falcon 9 Block 5 | Bandwagon 5 (Dedicated Mid-...,2026-10-20 11:04:00.000 UTC,1,100.0
7,H3-24 | Martian Moons eXploration (MMX),2026-10-19 19:41:03.000 UTC,1,100.0
8,Falcon 9 Block 5 | USSF-481,2026-10-16 00:01:05.000 UTC,2,100.0
9,Falcon 9 Block 5 | USSF-481,2026-10-15 23:57:05.000 UTC,1,0.0


,mission_name,launch_time,total_launches,success_rate
0,Long March 8A | SatNet LEO Group 28,2026-10-25 02:03:00.000 UTC,1,100.0
1,Falcon 9 Block 5 | Starlink Group 15-15,2026-10-23 22:48:44.000 UTC,2,100.0
2,Long March 4B | Yaogan 53-07 to 09/56-19 to 21,2026-10-23 19:25:00.000 UTC,2,100.0
3,Long March 6A | Yaogan 50-03,2026-10-23 10:15:00.000 UTC,2,100.0
4,Electron | We're Not Hawking Around (3x HawkEy...,2026-10-23 07:11:00.000 UTC,2,100.0
5,Ceres-1S | Xiguang-4 01,2026-10-21 03:39:00.000 UTC,2,100.0
6,Falcon 9 Block 5 | Bandwagon 5 (Dedicated Mid-...,2026-10-20 11:04:00.000 UTC,1,100.0
7,H3-24 | Martian Moons eXploration (MMX),2026-10-19 19:41:03.000 UTC,1,100.0
8,Falcon 9 Block 5 | USSF-481,2026-10-16 00:01:05.000 UTC,2,100.0
9,Falcon 9 Block 5 | USSF-481,2026-10-15 23:57:05.000 UTC,1,0.0
